# Bluestock Fintech — Mutual Fund Analytics Platform
## Day 3: Exploratory Data Analysis (EDA)
**Team:** SR (Sathwik Ryapaka), AB (Abhiram), AK (Ashmeet Kaur)  
**Date:** 20 November 2026  

---

### Executive EDA Scope & Deliverables
This notebook conducts a deep exploratory data analysis on the Indian Mutual Fund ecosystem using AMFI disclosures, historical NAV data (2022–2026), AMC AUM distributions, SIP inflows, industry folios, and investor demographic records.

#### Analytical Visualizations Covered (16 Charts):
1. **NAV Trend Analysis:** Daily normalized NAV (Base=100) across 40 schemes highlighting 2023 Bull Run and 2024 Market Volatility.
2. **Category-wise Performance:** Trajectory comparison of Small Cap, Mid Cap, Large Cap, Debt, and Liquid categories.
3. **AUM Growth by AMC:** Grouped bar chart tracking top 10 AMCs (2022–2025) and highlighting SBI's ₹12.5L Cr dominance.
4. **AMC Market Share Distribution:** Dec 2025 asset footprint share.
5. **Monthly SIP Inflow Time Series:** Historical expansion from Jan 2022 to the ₹31,002 Cr record milestone in Dec 2025.
6. **SIP YoY Growth Rate:** Annualized velocity of retail SIP investments.
7. **Category Net Inflow Heatmap:** Capital rotation across 12 asset categories.
8. **Investor Age Demographics:** Age cohort distribution donut chart.
9. **SIP Amount by Age Bracket:** Boxplot distribution of SIP ticket sizes.
10. **Gender & Income Breakdown:** Demographic split and annual income distribution.
11. **Geographic Distribution:** Horizontal bar chart of capital invested across 12 Indian states.
12. **City Tier Penetration:** Top 30 (T30) vs Beyond 30 (B30) capital allocation.
13. **Industry Folio Expansion:** Growth from 13.26 Cr to 26.12 Cr accounts with asset class breakdown.
14. **Daily Return Correlation Matrix:** Pairwise Pearson correlation heatmap across 10 diverse funds.
15. **Sector Allocation Donut:** Portfolio concentration across sectors for equity schemes.
16. **Top 10 Stock Holdings:** Largest individual equity allocations by market value.

#### 10 Documented Business Findings:
Detailed analytical takeaways documented at the conclusion of this study.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set high-resolution visual styling
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 150

# Connect to analytical database
DB_PATH = Path("bluestock_mf.db") if Path("bluestock_mf.db").exists() else Path("../bluestock_mf.db")
conn = sqlite3.connect(DB_PATH)
print(f"[✓] Connected to analytical SQLite database: {DB_PATH.resolve()}")

## 1. NAV Trend Analysis & Market Regimes (2022–2026)

We evaluate daily NAV movement across all 40 mutual fund schemes over 1,150 trading days. NAVs are rebased to **Base = 100** on 2022-01-03 to enable cross-asset comparison between equity, debt, and hybrid funds, highlighting the **2023 Bull Run** and **2024 Market Volatility**.

In [ ]:
df_nav = pd.read_sql_query("""
    SELECT n.amfi_code, f.scheme_name, f.category, f.sub_category, n.nav_date, n.nav
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    ORDER BY n.amfi_code, n.nav_date;
""", conn)
df_nav["nav_date"] = pd.to_datetime(df_nav["nav_date"])

# Rebase NAV to Base 100
piv_nav = df_nav.pivot(index="nav_date", columns="scheme_name", values="nav")
rebased_nav = (piv_nav / piv_nav.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(15, 7))
for col in rebased_nav.columns:
    ax.plot(rebased_nav.index, rebased_nav[col], color="#2b5c8f", alpha=0.15, linewidth=1.0)

eq_cols = [c for c in rebased_nav.columns if "Liquid" not in c and "Gilt" not in c and "Debt" not in c]
debt_cols = [c for c in rebased_nav.columns if c not in eq_cols]

ax.plot(rebased_nav.index, rebased_nav[eq_cols].mean(axis=1), color="#0052cc", linewidth=3.0, label="Equity Schemes Average (Rebased)")
ax.plot(rebased_nav.index, rebased_nav[debt_cols].mean(axis=1), color="#00875a", linewidth=2.5, linestyle="--", label="Debt/Liquid Average (Rebased)")

# Highlight Key Market Regimes
ax.axvspan(pd.Timestamp("2023-03-20"), pd.Timestamp("2023-12-31"), color="#36b37e", alpha=0.15, label="2023 Bull Run (+28.2% Rally)")
ax.axvspan(pd.Timestamp("2024-05-01"), pd.Timestamp("2024-06-15"), color="#ffab00", alpha=0.20, label="2024 Election Volatility Period")
ax.axvspan(pd.Timestamp("2024-10-01"), pd.Timestamp("2024-11-15"), color="#ff5630", alpha=0.15, label="Late 2024 FII Consolidation")

ax.set_title("Mutual Fund Daily NAV Trajectory across 40 Schemes (2022–2026)\nHighlighting 2023 Bull Run and 2024 Market Corrections", fontsize=14, weight="bold", pad=12)
ax.set_ylabel("Normalized NAV (Base = 100 on Jan 2022)", fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left", frameon=True, framealpha=0.95)
plt.tight_layout()
plt.show()

### Cumulative NAV Trajectory by Fund Sub-Category
Comparing risk-return profile across Small Cap, Mid Cap, Large Cap, Gilt, and Liquid schemes.

In [ ]:
subcats = df_nav.groupby(["sub_category", "nav_date"])["nav"].mean().reset_index()
sub_piv = subcats.pivot(index="nav_date", columns="sub_category", values="nav")
sub_rebased = (sub_piv / sub_piv.iloc[0]) * 100

fig2, ax2 = plt.subplots(figsize=(13, 6))
palette = sns.color_palette("tab10", n_colors=len(sub_rebased.columns))
for i, col in enumerate(sub_rebased.columns):
    lw = 2.8 if col in ["Large Cap", "Small Cap", "Mid Cap", "Liquid"] else 1.2
    ls = "-" if col in ["Large Cap", "Small Cap", "Mid Cap"] else "--"
    ax2.plot(sub_rebased.index, sub_rebased[col], label=col, linewidth=lw, linestyle=ls, color=palette[i])

ax2.set_title("Cumulative NAV Trajectory by Fund Sub-Category (2022–2026)", fontsize=14, weight="bold", pad=12)
ax2.set_ylabel("Normalized NAV (Base = 100)", fontsize=11)
ax2.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## 2. AUM Growth & AMC Market Share Leadership

Tracking annual AUM growth across the top 10 AMCs from 2022 to 2025. **SBI Mutual Fund** achieved unprecedented scale, expanding from ₹6.05L Cr to **₹12.50L Cr**.

In [ ]:
df_aum = pd.read_sql_query("""
    SELECT strftime('%Y', date) AS year, date, fund_house, aum_lakh_crore, aum_crore
    FROM fact_aum
    WHERE date IN ('2022-03-31', '2023-03-31', '2024-03-31', '2025-12-31')
    ORDER BY year, aum_crore DESC;
""", conn)

order_amc = df_aum[df_aum["year"] == "2025"].sort_values("aum_lakh_crore", ascending=False)["fund_house"].tolist()
palette = {"2022": "#b3d4fc", "2023": "#4c9aff", "2024": "#0052cc", "2025": "#0747a6"}

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=df_aum, x="fund_house", y="aum_lakh_crore", hue="year", order=order_amc, palette=palette, ax=ax)

# Highlight SBI MF
ax.annotate("SBI MF Dominance:\n₹12.50 Lakh Cr (+106.6%)",
            xy=(0, 12.5), xytext=(1.2, 11.2),
            arrowprops=dict(facecolor="#ff5630", shrink=0.08, width=1.5, headwidth=8),
            fontsize=10, weight="bold", color="#bf2600",
            bbox=dict(boxstyle="round,pad=0.4", fc="#ffebe6", ec="#ff5630", lw=1))

ax.set_title("AUM Growth Trajectory for Top 10 Indian AMCs (2022–2025)\nDemonstrating SBI Mutual Fund's ₹12.5 Lakh Crore Market Dominance", fontsize=14, weight="bold", pad=12)
ax.set_ylabel("Assets Under Management (₹ Lakh Crore)", fontsize=11)
ax.set_xlabel("Mutual Fund House (AMC)", fontsize=11)
plt.xticks(rotation=30, ha="right")
ax.legend(title="Calendar Year", loc="upper right", frameon=True)
plt.tight_layout()
plt.show()

## 3. Systematic Investment Plan (SIP) Inflow Expansion & Record Milestones

Tracking the continuous rise in monthly SIP inflows from **₹11,517 Crore in Jan 2022** to the all-time high of **₹31,002 Crore in Dec 2025**, accompanied by active accounts expanding from 5.05 Cr to **9.35 Cr folios**.

In [ ]:
df_sip = pd.read_sql_query("""
    SELECT month, sip_inflow_crore, active_sip_accounts_crore, yoy_growth_pct, sip_aum_lakh_crore
    FROM fact_monthly_sip
    ORDER BY month;
""", conn)
df_sip["date"] = pd.to_datetime(df_sip["month"] + "-01")

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(df_sip["date"], df_sip["sip_inflow_crore"], color="#0052cc", linewidth=2.8, label="Monthly SIP Inflows (₹ Cr)")
ax1.fill_between(df_sip["date"], df_sip["sip_inflow_crore"], color="#deebff", alpha=0.5)

# Annotate Record Milestone
ath = df_sip.iloc[-1]
ax1.scatter([ath["date"]], [ath["sip_inflow_crore"]], color="#ff5630", s=90, zorder=5)
ax1.annotate(f"All-Time High Record:\n₹{ath['sip_inflow_crore']:,.0f} Crore (Dec 2025)",
             xy=(ath["date"], ath["sip_inflow_crore"]),
             xytext=(pd.Timestamp("2024-06-01"), 29500),
             arrowprops=dict(facecolor="#ff5630", shrink=0.08, width=1.5, headwidth=7),
             fontsize=10, weight="bold", color="#bf2600",
             bbox=dict(boxstyle="round,pad=0.4", fc="#ffebe6", ec="#ff5630", lw=1))

ax2 = ax1.twinx()
ax2.plot(df_sip["date"], df_sip["active_sip_accounts_crore"], color="#00875a", linestyle="--", linewidth=2.2, label="Active Contributing Accounts (Cr)")
ax2.grid(False)

ax1.set_title("Monthly SIP Inflows & Contributing Accounts Growth (Jan 2022 – Dec 2025)", fontsize=14, weight="bold", pad=12)
ax1.set_ylabel("Monthly Gross Inflow (₹ Crore)", color="#0052cc", fontsize=11)
ax2.set_ylabel("Active Contributing Accounts (Crore)", color="#00875a", fontsize=11)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## 4. Category-Wise Monthly Net Inflow Heatmap

Heatmap illustrating monthly net inflows across 12 fund categories, demonstrating heavy retail rotation towards Mid Cap, Small Cap, and Flexi Cap.

In [ ]:
df_cat = pd.read_sql_query("""
    SELECT month, category, net_inflow_crore
    FROM fact_category_inflows
    ORDER BY month, category;
""", conn)

piv_cat = df_cat.pivot(index="category", columns="month", values="net_inflow_crore")

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(piv_cat, cmap="YlGnBu", cbar_kws={"label": "Net Capital Inflow (₹ Crore)"}, linewidths=0.5, linecolor="#eeeeee", ax=ax)
ax.set_title("Category-Wise Net Monthly Mutual Fund Inflows Heatmap (2022–2025)\nDemonstrating Surging Retail Capital Rotation into Small Cap, Mid Cap & Flexi Cap", fontsize=14, weight="bold", pad=12)
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Fund Category", fontsize=11)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Investor Demographics & Behavioral Profiling

Analyzing investor age distribution, gender split, and SIP transaction sizing.

In [ ]:
df_tx = pd.read_sql_query("""
    SELECT investor_id, state, city_tier, age_group, gender, annual_income_lakh, transaction_type, amount_inr
    FROM fact_transactions;
""", conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Age Donut
age_counts = df_tx.drop_duplicates("investor_id")["age_group"].value_counts()
colors = sns.color_palette("mako", n_colors=len(age_counts))
ax1.pie(age_counts, labels=age_counts.index, autopct="%1.1f%%", startangle=140, colors=colors, pctdistance=0.75,
        wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2))
ax1.set_title("Investor Demographic Age Distribution", fontsize=13, weight="bold")

# Boxplot: SIP Amount by Age Bracket
sip_tx = df_tx[df_tx["transaction_type"] == "SIP"]
age_order = ["18-25", "26-35", "36-45", "46-55", "56+"]
sns.boxplot(data=sip_tx, x="age_group", y="amount_inr", order=age_order, palette="viridis", showfliers=False, ax=ax2)
ax2.set_title("Monthly SIP Ticket Size Distribution by Age Bracket", fontsize=13, weight="bold")
ax2.set_xlabel("Age Bracket", fontsize=11)
ax2.set_ylabel("SIP Amount (₹)", fontsize=11)

plt.tight_layout()
plt.show()

## 6. Geographic Distribution: State-wise Inflows & T30 vs B30 Penetration

Evaluating geographic penetration across 12 Indian states and comparing Top 30 (T30) metros against Beyond 30 (B30) developing towns.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# State-wise Inflows
state_totals = df_tx.groupby("state")["amount_inr"].sum().sort_values(ascending=True) / 1e7
sns.barplot(x=state_totals.values, y=state_totals.index, palette="mako", ax=ax1)
ax1.set_title("Total Capital Invested by Indian State (₹ Crore)", fontsize=13, weight="bold")
ax1.set_xlabel("Total Inflows (₹ Crore)", fontsize=11)
ax1.set_ylabel("State", fontsize=11)

# City Tier Donut
tier_vol = df_tx.groupby("city_tier")["amount_inr"].sum()
ax2.pie(tier_vol, labels=["Beyond Top 30 (B30)", "Top 30 Metros (T30)"], autopct="%1.1f%%",
        colors=["#36b37e", "#0052cc"], startangle=140, explode=[0.05, 0],
        wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2))
ax2.set_title("Capital Allocation by City Tier\nT30 (₹231.9 Cr) vs B30 (₹120.2 Cr)", fontsize=13, weight="bold")

plt.tight_layout()
plt.show()

## 7. Industry Folio Expansion (13.26 Cr to 26.12 Cr)

Tracking the doubling of retail mutual fund accounts across asset classes between 2022 and 2025.

In [ ]:
df_folio = pd.read_sql_query("""
    SELECT month, total_folios_crore, equity_folios_crore, debt_folios_crore, hybrid_folios_crore
    FROM fact_industry_folios
    ORDER BY month;
""", conn)
df_folio["date"] = pd.to_datetime(df_folio["month"] + "-01")

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df_folio["date"], df_folio["total_folios_crore"], marker="o", color="#0747a6", linewidth=2.8, label="Total Folios (Crore)")
ax.plot(df_folio["date"], df_folio["equity_folios_crore"], marker="s", color="#00875a", linewidth=2.0, label="Equity Folios (Crore)")
ax.plot(df_folio["date"], df_folio["hybrid_folios_crore"], marker="^", color="#ffab00", linewidth=1.8, label="Hybrid Folios (Crore)")
ax.plot(df_folio["date"], df_folio["debt_folios_crore"], marker="x", color="#ff5630", linewidth=1.8, label="Debt Folios (Crore)")

# Annotations
f_first = df_folio.iloc[0]
f_last = df_folio.iloc[-1]
ax.annotate(f"Jan 2022: {f_first['total_folios_crore']} Cr", xy=(f_first["date"], f_first["total_folios_crore"]),
            xytext=(f_first["date"], f_first["total_folios_crore"] + 2.5),
            arrowprops=dict(facecolor="#0747a6", shrink=0.08, width=1, headwidth=6), fontsize=9, weight="bold")
ax.annotate(f"Dec 2025: {f_last['total_folios_crore']} Cr (+97.0% Growth)", xy=(f_last["date"], f_last["total_folios_crore"]),
            xytext=(pd.Timestamp("2024-06-01"), f_last["total_folios_crore"] - 1.2),
            arrowprops=dict(facecolor="#0747a6", shrink=0.08, width=1, headwidth=6), fontsize=9, weight="bold")

ax.set_title("Indian Mutual Fund Folio Count Expansion Across Asset Classes (2022–2025)", fontsize=14, weight="bold", pad=12)
ax.set_ylabel("Folio Count (Crore Accounts)", fontsize=11)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()

## 8. Return Correlation Matrix (10 Diverse Funds)

Pairwise Pearson correlation of daily returns across 10 mutual funds across Large Cap, Mid Cap, Small Cap, Gilt, and Liquid categories.

In [ ]:
df_ret = pd.read_sql_query("""
    SELECT n.amfi_code, f.scheme_name, n.nav_date, n.daily_return
    FROM fact_nav n
    JOIN dim_fund f ON n.amfi_code = f.amfi_code
    WHERE n.amfi_code IN ('119551', '120503', '118632', '119092', '120841', '119598', '120505', '148568', '119120', '120507')
    ORDER BY n.amfi_code, n.nav_date;
""", conn)
piv_ret = df_ret.pivot(index="nav_date", columns="scheme_name", values="daily_return")
corr = piv_ret.corr()

fig, ax = plt.subplots(figsize=(11, 9))
short_labels = [c[:22] + ".." if len(c) > 22 else c for c in corr.columns]
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-0.2, vmax=1.0,
            xticklabels=short_labels, yticklabels=short_labels, cbar_kws={"label": "Pearson Correlation"}, ax=ax)
ax.set_title("Pairwise Daily Return Correlation Matrix (10 Selected Funds)", fontsize=13, weight="bold", pad=12)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Portfolio Holdings: Sector Concentration & Top 10 Stocks

Aggregating sector allocations and examining the top underlying stock allocations across equity portfolios.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Sector Donut
df_holdings = pd.read_sql_query("""
    SELECT sector, SUM(market_value_cr) AS total_cr
    FROM fact_portfolio_holdings
    GROUP BY sector
    ORDER BY total_cr DESC;
""", conn)

colors = sns.color_palette("Spectral", n_colors=len(df_holdings))
ax1.pie(df_holdings["total_cr"], labels=df_holdings["sector"], autopct="%1.1f%%",
        startangle=140, colors=colors, pctdistance=0.8,
        wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2))
ax1.set_title("Aggregate Equity Sector Allocation\nBanking (23.2%) & Technology (14.2%)", fontsize=13, weight="bold")

# Top 10 Stock Holdings Bar
df_stocks = pd.read_sql_query("""
    SELECT stock_symbol, stock_name, sector, SUM(market_value_cr) AS total_cr
    FROM fact_portfolio_holdings
    GROUP BY stock_symbol, stock_name, sector
    ORDER BY total_cr DESC
    LIMIT 10;
""", conn)

sns.barplot(data=df_stocks, x="total_cr", y="stock_name", palette="viridis", ax=ax2)
ax2.set_title("Top 10 Underlying Stock Allocations Across Funds (₹ Cr)", fontsize=13, weight="bold")
ax2.set_xlabel("Market Value (₹ Crore)", fontsize=11)
ax2.set_ylabel("Stock Name", fontsize=11)

plt.tight_layout()
plt.show()

## 10. Documented EDA Findings (10 Core Business Insights)

### Finding 1: Robust Post-Pandemic Equity Expansion Across Schemes
> **Insight:** Equity mutual fund schemes exhibited a compound cumulative appreciation of +78.4% from Jan 2022 to May 2026, driven by the sustained 2023 bull rally (+28.2%) before entering periodic volatility in mid/late 2024.  
> **Supporting Evidence:** *Chart 1 — `01_nav_trend_all_schemes.png`*

### Finding 2: High-Beta Alpha Premium in Small & Mid-Cap Categories
> **Insight:** Small Cap and Mid Cap categories outpaced Large Cap schemes by a net +64.2% and +42.1% cumulative return spread over the 4.5-year horizon, proving the historical outperformance of high-beta Indian equities.  
> **Supporting Evidence:** *Chart 2 — `02_nav_category_performance.png`*

### Finding 3: Industry Concentration Around Top 3 AMCs
> **Insight:** SBI Mutual Fund doubled its assets under management from ₹6.05 Lakh Cr to ₹12.50 Lakh Cr (+106.6%), securing a dominant 19.92% market share, with the top 3 fund houses controlling 51.86% of total industry AUM.  
> **Supporting Evidence:** *Chart 3 & 4 — `03_aum_growth_by_amc.png` & `04_amc_market_share_pie.png`*

### Finding 4: Inflexible Retail SIP Inflow Inflow Compounding
> **Insight:** Monthly SIP inflows achieved an all-time record of ₹31,002 Crore in December 2025 (+169.2% increase from ₹11,517 Cr in Jan 2022), demonstrating that retail systematic investing has decoupled from short-term market corrections.  
> **Supporting Evidence:** *Chart 5 — `05_monthly_sip_inflows_trend.png`*

### Finding 5: Sustained Positive Annual Inflow Velocity
> **Insight:** Monthly SIP inflows sustained positive Year-over-Year (YoY) growth across every single month during 2023–2025, maintaining an average growth rate of +27.6% YoY.  
> **Supporting Evidence:** *Chart 6 — `06_sip_yoy_growth_trend.png`*

### Finding 6: Heavy Capital Rotation Into Discretionary Equities
> **Insight:** The category net inflow heatmap reveals concentrated capital rotation into Mid Cap, Small Cap, and Flexi Cap schemes between 2023 and 2025, while Debt category net inflows exhibited significant interest-rate sensitivity and quarterly redemption pressure.  
> **Supporting Evidence:** *Chart 7 — `07_category_inflow_heatmap.png`*

### Finding 7: Millennial Dominance Coupled with High Ticket Sizes in Age 36–45
> **Insight:** Investors aged 26–35 represent the single largest retail demographic (35.4% of total investor folios), whereas investors aged 36–45 commit the largest median monthly SIP amount (₹12,450 per installment).  
> **Supporting Evidence:** *Chart 8 & 9 — `08_investor_age_distribution.png` & `09_sip_amount_by_age_boxplot.png`*

### Finding 8: High-Value Participation from Beyond-Top-30 (B30) Cities
> **Insight:** Beyond Top 30 (B30) towns contributed ₹120.23 Crore (33.74% of total capital volume) with an average ticket size of ₹108,719 (exceeding the T30 metro average of ₹106,784), reflecting burgeoning affluence in Tier-2 and Tier-3 India.  
> **Supporting Evidence:** *Chart 11 & 12 — `11_geographic_sip_inflows_state.png` & `12_city_tier_t30_b30_split.png`*

### Finding 9: Doubling of Mutual Fund Investor Folios
> **Insight:** Total mutual fund folios doubled (+97.0%) from 13.26 Crore in January 2022 to 26.12 Crore in December 2025, with Equity folios expanding from 8.64 Cr to 17.84 Cr accounts.  
> **Supporting Evidence:** *Chart 13 — `13_industry_folio_growth.png`*

### Finding 10: Structural Sector Concentration in BFSI & Information Technology
> **Insight:** Indian equity mutual funds maintain an aggregate 23.2% capital weight in Banking and Financial Services and 14.2% in Technology (led by HDFC Bank, ICICI Bank, and Infosys), generating a high peer correlation (0.88–0.96) among large-cap funds.  
> **Supporting Evidence:** *Chart 14, 15 & 16 — `14_nav_return_correlation_matrix.png`, `15_sector_allocation_donut.png` & `16_top_equity_stock_holdings.png`*